# Lesson 04: Using Generative AI Responsibly

In this notebook, you will:
1. Configure `google-genai` Layer 2 `SafetySetting` policies (`HarmCategory` + `HarmBlockThreshold`).
2. Apply Layer 3 Grounded Metaprompts to defeat premise-trap hallucinations like *"Who was the sole survivor of the Titanic?"*.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os
from google import genai
from google.genai import types

def get_client():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    key = key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=key) if key else None

client = get_client()

In [ ]:
prompt = "Who was the sole survivor of the Titanic?"

config = types.GenerateContentConfig(
    temperature=0.1,
    system_instruction=(
        "You are a rigorous historical fact-checker. If a question contains a false premise, "
        "explicitly state why the premise is false before providing the verified facts."
    ),
    safety_settings=[
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
    ],
)

if client:
    resp = client.models.generate_content(model="gemini-2.5-flash", contents=prompt, config=config)
    print(resp.text)
else:
    print("[Offline Demo Mode] Premise corrected: There was no single survivor. Over 700 people survived the Titanic disaster in 1912.")